## Exercise 6: Shared utility functions, data catalogs

Skills: 
* Import shared utils
* Data catalog
* Use functions to repeat certain data cleaning steps

References: 
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_tools/data_catalogs.html

In [1]:
import geopandas as gpd
import intake
import pandas as pd
pd.options.display.max_rows = 100
pd.options.display.float_format = "{:.0f}".format
#commas to separate 1000s 
pd.options.display.float_format = '{:,}'.format
# Hint: if this doesn't import: refer to docs for correctly import
# cd into _shared_utils folder, run the make setup_env command
import shared_utils

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.9.1-CAPI-1.14.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


## Create a data catalog - HELP can't read open data portal stuff.

* Include one geospatial data source and one tabular (they should be related...your analysis depends on combining them)
* Import your datasets using the catalog method

Amanda - To-Do
* Change processed TIRCP dataframe to excel from CSV to preserve data types.

In [2]:
#Import Catalog: not working.
#catalog = intake.open_catalog("./Catalog.yml")

In [3]:
tircp = pd.read_csv('gs://calitp-analytics-data/data-analyses/tircp/df_tableau_sheet.csv')

In [4]:
#only keep columns I want. Drop NA for district, filter out projects with districts coded as "various"
tircp = (tircp[['Award_Year', 'Local_Agency','District','Local_Agency_City','TIRCP_Amount', 'County','Project_Title']]
         .loc[tircp['District'] != 'Various'].dropna(subset=['District']))

In [5]:
tircp.shape

(58, 7)

In [6]:
tircp

,Award_Year,Local_Agency,District,Local_Agency_City,TIRCP_Amount,County,Project_Title
0,2015,Antelope Valley Transit Authority (AVTA),District 7: Los Angeles,Lancaster,"24,403,000.0",LA,Regional Transit Interconnectivity & Environme...
1,2015,Capitol Corridor Joint Powers Authority,District 4: Bay Area / Oakland,PO Box 12688\nOakland,"4,620,000.0",VAR,Travel Time Reduction Project
2,2015,Los Angeles County Metropolitan Transportation...,District 7: Los Angeles,Los Angeles,"38,494,000.0",LA,Willowbrook/Rosa Parks Station & Blue Line Lig...
4,2015,Montery-Salinas Transit,District 5: San Luis Obispo / Santa Barbara,Monterey,"10,000,000.0",MON,Monterey Bay Operations and Maintenance Facili...
5,2015,Orange County Transportation Authority (OCTA),District 12: Orange County,Orange,"2,320,000.0",ORA,Bravo! Route 560 Rapid Buses
6,2015,Sacramento Regional Transit District (SacRT),District 3: Marysville / Sacramento,Sacramento,"6,427,000.0",SAC,Sacramento Regional Transit's Refurbishment of...
7,2015,San Diego Association of Governments (SANDAG),District 11: San Diego,San Diego,"4,000,000.0",SD,South Bay Bus Rapid Transit
8,2015,San Diego Metropolitan Transit System (MTS),District 11: San Diego,San Diego,"31,986,000.0",SD,San Diego Metropolitan Transit System Trolley ...
9,2015,San Francisco Municipal Transportation Agency,District 4: Bay Area / Oakland,San Francisco,"41,181,000.0",SF,SFMTA Light Rail Vehicle Fleet Expansion
10,2015,San Joaquin Regional Rail Commission / San Joa...,District 10: Stockton,Stockton,"200,000.0",SJ,ACE Wayside Power Project


In [7]:
tircp.District.value_counts()

District 7: Los Angeles                        17
District 4: Bay Area / Oakland                 14
District 11: San Diego                          6
District 3: Marysville / Sacramento             5
District 10: Stockton                           4
District 5: San Luis Obispo / Santa Barbara     3
District 12: Orange County                      3
District 8: San Bernardino / Riverside          3
District 6: Fresno / Bakersfield                2
District 1: Eureka                              1
Name: District, dtype: int64

In [17]:
tircp.dtypes

Award_Year             int64
Local_Agency          object
District              object
Local_Agency_City     object
TIRCP_Amount         float64
County                object
Project_Title         object
dtype: object

In [8]:
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/e397d899e7be4ce28ad261867e61ac69_0.geojson')

In [9]:
#Renaming districts in geojson file.
geojson['DISTRICT'] = (geojson['DISTRICT'].replace({7:'District 7: Los Angeles',
                                                    2: 'District 2: Redding',
                                            4:'District 4: Bay Area / Oakland',
                                            'VAR':'Various',
                                            10:'District 10: Stockton',
                                            11:'District 11: San Diego',
                                            3:'District 3: Marysville / Sacramento',
                                            12: 'District 12: Orange County',
                                            8: 'District 8: San Bernardino / Riverside',
                                            5:'District 5: San Luis Obispo / Santa Barbara',
                                            9: 'District 9: Bishop',
                                            6:'District 6: Fresno / Bakersfield',
                                            1:'District 1: Eureka'
                                                 }))
    

In [10]:
geojson = geojson[['DISTRICT','Shape_Length','Shape_Area','geometry']]

In [11]:
geojson

,DISTRICT,Shape_Length,Shape_Area,geometry
0,District 1: Eureka,"1,913,828.842199962","41,884,999,595.1419","POLYGON ((-123.51819 42.00031, -123.51794 42.0..."
1,District 2: Redding,"1,776,127.9212410385","126,685,500,472.3809","POLYGON ((-122.30683 42.00816, -122.32039 42.0..."
2,District 3: Marysville / Sacramento,"1,467,236.2805911216","54,594,706,314.0322","POLYGON ((-121.40463 40.14665, -121.40452 40.1..."
3,District 4: Bay Area / Oakland,"1,540,172.0382685345","31,232,835,388.638123","POLYGON ((-122.35190 38.83627, -122.35432 38.8..."
4,District 5: San Luis Obispo / Santa Barbara,"1,593,306.215365071","43,412,204,414.475716","POLYGON ((-122.15057 37.27596, -122.15058 37.2..."
5,District 6: Fresno / Bakersfield,"1,457,489.187639208","77,688,824,926.97177","POLYGON ((-119.27249 37.73954, -119.27299 37.7..."
6,District 8: San Bernardino / Riverside,"1,365,819.025713105","104,915,788,285.77463","POLYGON ((-115.66384 35.80924, -115.68169 35.8..."
7,District 10: Stockton,"1,199,089.6727459044","45,919,540,649.088326","POLYGON ((-119.88972 38.92320, -119.89712 38.9..."
8,District 11: San Diego,"1,104,658.5557319298","32,215,760,156.229317","POLYGON ((-117.36428 33.50503, -117.43208 33.5..."
9,District 12: Orange County,"361,134.2818145595","2,983,890,308.6631484","MULTIPOLYGON (((-117.70373 33.46011, -117.7039..."


In [12]:
geojson['DISTRICT'] = geojson.astype({'DISTRICT': 'str'}).dtypes

In [13]:
geojson.dtypes

DISTRICT          object
Shape_Length     float64
Shape_Area       float64
geometry        geometry
dtype: object

In [14]:
type(geojson)

geopandas.geodataframe.GeoDataFrame

## Combine datasets - HELP 
* How do I join a normal dataframe with a geo dataframe? Right now, I have to do a left join on the geo dataframe which isnt what I want: I want my TIRCP grants df to be on the left.
* Why did my district disappear once I joined everything?

Tiffany's Notes
* Do a merge or spatial join to combine the geospatial and tabular data
* Create a new column of a summary statistic to visualize
* Rely on `shared_utils` to do at least one operation (aggregation, re-projecting to a different CRS, exporting geoparquet, etc)

In [15]:
#Joining the 2 dataframes, only keeping stops that are in a county
joined = geojson.merge(tircp, how = 'right', left_on ='DISTRICT', right_on = 'District')

In [16]:
joined

,DISTRICT,Shape_Length,Shape_Area,geometry,Award_Year,Local_Agency,District,Local_Agency_City,TIRCP_Amount,County,Project_Title
0,NaN,NaN,NaN,None,2015,Antelope Valley Transit Authority (AVTA),District 7: Los Angeles,Lancaster,"24,403,000.0",LA,Regional Transit Interconnectivity & Environme...
1,NaN,NaN,NaN,None,2015,Capitol Corridor Joint Powers Authority,District 4: Bay Area / Oakland,PO Box 12688\nOakland,"4,620,000.0",VAR,Travel Time Reduction Project
2,NaN,NaN,NaN,None,2015,Los Angeles County Metropolitan Transportation...,District 7: Los Angeles,Los Angeles,"38,494,000.0",LA,Willowbrook/Rosa Parks Station & Blue Line Lig...
3,NaN,NaN,NaN,None,2015,Montery-Salinas Transit,District 5: San Luis Obispo / Santa Barbara,Monterey,"10,000,000.0",MON,Monterey Bay Operations and Maintenance Facili...
4,NaN,NaN,NaN,None,2015,Orange County Transportation Authority (OCTA),District 12: Orange County,Orange,"2,320,000.0",ORA,Bravo! Route 560 Rapid Buses
5,NaN,NaN,NaN,None,2015,Sacramento Regional Transit District (SacRT),District 3: Marysville / Sacramento,Sacramento,"6,427,000.0",SAC,Sacramento Regional Transit's Refurbishment of...
6,NaN,NaN,NaN,None,2015,San Diego Association of Governments (SANDAG),District 11: San Diego,San Diego,"4,000,000.0",SD,South Bay Bus Rapid Transit
7,NaN,NaN,NaN,None,2015,San Diego Metropolitan Transit System (MTS),District 11: San Diego,San Diego,"31,986,000.0",SD,San Diego Metropolitan Transit System Trolley ...
8,NaN,NaN,NaN,None,2015,San Francisco Municipal Transportation Agency,District 4: Bay Area / Oakland,San Francisco,"41,181,000.0",SF,SFMTA Light Rail Vehicle Fleet Expansion
9,NaN,NaN,NaN,None,2015,San Joaquin Regional Rail Commission / San Joa...,District 10: Stockton,Stockton,"200,000.0",SJ,ACE Wayside Power Project


## Use functions to do parameterized visualizations
* Use a function to create your chart
* Within the function, the colors should use the Cal-ITP theme that is available in `styleguide`
* Within the function, there should be at least 1 parameter that changes (ex: chart title reflects the correct county, legend title reflects the correct county, etc)
* Produce 3 charts, using your function each time, and have the function correctly insert the parameters 